In [1]:
import duckdb
import pandas as pd
from pathlib import Path
import numpy as np

class DataHubReader:
    def __init__(self, db_name="silver_company.db"):
        """
        Stellt Verbindung zur DuckDB her.
        Erwartet dieselbe Ordnerstruktur wie DataHub.
        """
        base_path = Path.cwd()
        db_path = base_path.parent.parent / "data"  / db_name

        if not db_path.exists():
            raise FileNotFoundError(f"Datenbank nicht gefunden: {db_path}")
        self.con = duckdb.connect(str(db_path))

    def _fetch_table(self, table_name: str) -> pd.DataFrame:
        """Generische Methode zum Laden einer Tabelle."""
        return self.con.execute(f"SELECT * FROM {table_name}").df()
    def sql_command(self, command):
        return self.con.execute(f'{command}').df()
    def get_prices(self):
        return self.con.execute(f'Select * from observations').df()
    def close(self):
        """Schließt die Datenbankverbindung."""
        self.con.close()

In [ ]:
hub =  DataHubReader()
data = hub.sql_command("Select * from observations")
items =  hub.sql_command("Select * from items")
result = data.merge(items[["ItemID", "ItemCategory"]], on="ItemID", how="left")
fundamentals = result[result['ItemCategory'].isin(['balance_sheet', 'cashflow', 'income_statement'])]

In [20]:
fundamentals = fundamentals[fundamentals['Date'] <= '31.12.2025']
fundamentals['Date'] = pd.to_datetime(fundamentals['Date'].dt.year.astype(str) + '-12-31')

In [ ]:
total_companies = fundamentals['EntityCode'].unique().sum()
coverage = (
    data.groupby('ItemName')['EntityCode']
    .nunique()
    .reset_index()
    .rename(columns={'EntityCode': 'company_count'})
    .sort_values('company_count', ascending=False)
)
coverage['coverage_pct'] = coverage['company_count'] / total_companies * 100
coverage.head()

1267677